# Módulo 6 · Tres problemas típicos de ingesta

⏱ **11 min** · Slide: 29

**Objetivo:** provocar los tres problemas a propósito, verlos fallar y aplicar la solución.

| Parte | Problema | Solución |
|---|---|---|
| A | Archivos corruptos | Modo de lectura: `PERMISSIVE`, `DROPMALFORMED`, `FAILFAST` |
| B | El esquema cambia | Delta lo rechaza hasta que aceptes con `mergeSchema` |
| C | Duplicados al re-ejecutar | Cargas idempotentes: registrar archivos o `MERGE` |

Trabajamos con **tablas de prueba** (`*_demo_*`) para no tocar `bronze_episodios`.

**Requiere:** módulos 2 y 5.

In [0]:
%run ./00_Setup

# Parte A · Archivos corruptos

Creamos un CSV con dos filas malas:
- la fila `1003` trae `abc` donde debería haber un número
- la fila `1004` trae una columna de más

In [0]:
ruta_sucio = f"{TMP_PATH}/episodios_sucios.csv"

contenido = """episode_id,serie,temporada,minutos
1001,Dark Matter,1,45
1002,Foundation,2,50
1003,The Bear,3,abc
1004,Severance,2,55,COLUMNA_DE_MAS
1005,Fallout,1,60
"""
_ = dbutils.fs.put(ruta_sucio, contenido, True)

esquema_csv = "episode_id INT, serie STRING, temporada INT, minutos INT"

### A.1 · `PERMISSIVE` (por defecto): conserva todo y marca lo malo
Agregamos la columna especial `_corrupt_record` para ver qué línea falló.

In [0]:
permissive = (spark.read.option("header", True)
              .option("mode", "PERMISSIVE")
              .schema(esquema_csv + ", _corrupt_record STRING")
              .csv(ruta_sucio))
display(permissive)

### A.2 · `DROPMALFORMED`: descarta las filas malas en silencio

In [0]:
dropmalformed = (spark.read.option("header", True)
                 .option("mode", "DROPMALFORMED")
                 .schema(esquema_csv)
                 .csv(ruta_sucio))
display(dropmalformed)

### A.3 · `FAILFAST`: se detiene al primer error
El error aparece al ejecutar la **acción** (`collect`), no al definir la lectura. ¡Es el comportamiento *lazy* del módulo 3!

In [0]:
failfast = (spark.read.option("header", True)
            .option("mode", "FAILFAST")
            .schema(esquema_csv)
            .csv(ruta_sucio))

try:
    failfast.collect()
except Exception as e:
    print("❌ FAILFAST detuvo la carga:", str(e).splitlines()[0][:200])

👀 **Qué observar:**

| Modo | Filas buenas | Filas malas | Cuándo usarlo |
|---|---|---|---|
| `PERMISSIVE` | Se leen | Se conservan y se marcan en `_corrupt_record` | Bronze: no perder nada y auditar |
| `DROPMALFORMED` | Se leen | Se pierden sin aviso | Solo si perder datos es aceptable |
| `FAILFAST` | — | Aborta toda la carga | Datos críticos que no admiten errores |

# Parte B · El esquema cambia

Llega una columna nueva (`calidad_dato`) que la tabla no conoce.

In [0]:
DEMO_ESQUEMA = "bronze_demo_esquema"

spark.table(BRONZE).limit(100).write.mode("overwrite").saveAsTable(DEMO_ESQUEMA)
print("Columnas de la tabla:", len(spark.table(DEMO_ESQUEMA).columns))

In [0]:
nuevo_lote = spark.table(BRONZE).limit(10).withColumn("calidad_dato", F.lit("A"))

try:
    nuevo_lote.write.mode("append").saveAsTable(DEMO_ESQUEMA)
except Exception as e:
    print("❌ Delta rechazó la escritura:", str(e)[:250])

👀 **Qué observar:** Delta **protege** la tabla y no deja entrar columnas desconocidas por accidente (*schema enforcement*).
Si el cambio es legítimo, lo aceptas de forma explícita:

In [0]:
nuevo_lote.write.mode("append").option("mergeSchema", True).saveAsTable(DEMO_ESQUEMA)

print("Columnas ahora:", len(spark.table(DEMO_ESQUEMA).columns))
display(spark.table(DEMO_ESQUEMA)
        .groupBy(F.col("calidad_dato").isNull().alias("fila_antigua_sin_valor"))
        .count())

👀 **Qué observar:** la tabla ahora tiene la columna nueva; las 100 filas antiguas quedaron con `NULL` en ella.

# Parte C · Duplicados al re-ejecutar

Definimos tres formas de cargar y las ejecutamos **dos veces** cada una.

In [0]:
from delta.tables import DeltaTable

DEMO_IDEM = "bronze_demo_idempotencia"

def reset_table():
    spark.sql(f"DROP TABLE IF EXISTS {DEMO_IDEM}")

def row_count():
    return spark.table(DEMO_IDEM).count()

### C.1 · Carga ingenua (solo `append`)

In [0]:
def load_naive():
    read_raw().write.mode("append").saveAsTable(DEMO_IDEM)

reset_table()
load_naive()
print("Tras la 1.ª ejecución:", row_count())
load_naive()
print("Tras la 2.ª ejecución:", row_count(), " ← ¡el doble!")

### C.2 · Solución A: registrar los archivos ya cargados
Solo se cargan archivos cuyo `archivo_origen` aún no está en la tabla (`left_anti` join).

In [0]:
def load_without_repeats():
    nuevos = read_raw()
    if spark.catalog.tableExists(DEMO_IDEM):
        ya_cargados = spark.table(DEMO_IDEM).select("archivo_origen").distinct()
        nuevos = nuevos.join(ya_cargados, "archivo_origen", "left_anti")
    nuevos.write.mode("append").saveAsTable(DEMO_IDEM)

reset_table()
load_without_repeats()
print("Tras la 1.ª ejecución:", row_count())
load_without_repeats()
print("Tras la 2.ª ejecución:", row_count(), " ← igual")

### C.3 · Solución B: `MERGE` por clave
Inserta lo nuevo y actualiza lo que ya existe, usando `id` como clave.
Antes se eliminan claves repetidas **en el origen** (un `MERGE` falla si dos filas origen coinciden con la misma fila destino).

In [0]:
def load_merge():
    origen = read_raw().dropDuplicates(["id"])
    if not spark.catalog.tableExists(DEMO_IDEM):
        origen.write.saveAsTable(DEMO_IDEM)
        return
    (DeltaTable.forName(spark, qualified_name(DEMO_IDEM)).alias("t")
        .merge(origen.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

reset_table()
load_merge()
print("Tras la 1.ª ejecución:", row_count())
load_merge()
print("Tras la 2.ª ejecución:", row_count(), " ← igual")

👀 **Qué observar:** con A y B el conteo no cambia al repetir la carga.

| | Registrar archivos | `MERGE` |
|---|---|---|
| Evita duplicados | ✅ | ✅ |
| Actualiza filas que cambiaron en el origen | ❌ | ✅ |
| Costo | Bajo | Mayor |
| Ideal para | Archivos que llegan una vez y no cambian | Datos que se corrigen o reenvían |

## Limpieza de las tablas de prueba

In [0]:
for t in (DEMO_ESQUEMA, DEMO_IDEM):
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("✔ Tablas de prueba eliminadas. bronze_episodios no se tocó.")

## ✔ Checkpoint

- [ ] Distingo `PERMISSIVE`, `DROPMALFORMED` y `FAILFAST`
- [ ] Vi a Delta rechazar un cambio de esquema y lo acepté con `mergeSchema`
- [ ] Puedo explicar por qué `append` doble duplica y cómo evitarlo

**Siguiente:** `07_Silver_y_Gold`